# ResNet 与 CIFAR-10：完整训练与测试

使用 PyTorch 训练适配 `32×32` 彩色图像的 ResNet-18，完成 CIFAR-10 十分类。
模型、残差块、训练循环、验证、模型保存与测试代码均已提供。
按顺序运行各单元，查看训练曲线，加载最佳参数完成独立测试。

使用已有的 `cv-cnn` 环境，操作说明见 [README](README.md) 和
[实践任务](任务书_ResNet_CIFAR10实践课.md)。
初始资料不包含数据集、预训练参数或训练结果。


## 1. 导入依赖

内核选择 `Python (cv-cnn)`。从本项目目录启动 JupyterLab，保留资料包内的文件结构。


In [ ]:
from pathlib import Path
import json, os, re, sys
import numpy as np
import torch
import torchvision
from torch import nn
import matplotlib.pyplot as plt

LAB_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / 'resnet_common.py').is_file()), None)
if LAB_ROOT is None:
    raise FileNotFoundError('请从实践课_ResNet_CIFAR10目录启动 JupyterLab。')
sys.path.insert(0, str(LAB_ROOT))
from resnet_common import (
    CLASS_NAMES, CIFAR_MEAN, CIFAR_STD, set_seed, select_device,
    make_loaders, make_test_loader, evaluate, fit, count_parameters,
    show_batch, plot_history, plot_confusion, show_mistakes,
)
print('Python:', sys.version.split()[0], '| interpreter:', sys.executable)
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available(), '| MPS:', torch.backends.mps.is_available())


## 2. 设置运行配置

完整数据配置为训练 45,000 张、验证 5,000 张，训练 20 轮。
ResNet-18 的计算量明显大于简单 CNN，完整数据训练可使用兼容的 CUDA 或 MPS 设备。

使用 CPU 时，可先把 `TRAIN_SIZE, VAL_SIZE` 改为 `2000, 500`，
`EPOCHS` 改为 `3`，`RUN_NAME` 改为 `cpu_run`。
仅检查流程时可使用 `128, 64`、`1` 轮与名称 `smoke`。
小数据和少量轮次用于运行检查，所得准确率不能代表完整训练效果。

GPU 完整训练保持 45,000/5,000 张，将 `DEVICE_CHOICE` 改为 `cuda` 或 `mps`。
延长训练时可设置 `EPOCHS = 100` 并使用新名称。
`auto` 按 CUDA、MPS、CPU 的顺序选择设备，实际设备会打印出来。


In [ ]:
DEVICE_CHOICE = 'cpu'             # 'cpu' / 'cuda' / 'mps' / 'auto'
TRAIN_SIZE, VAL_SIZE = 45000, 5000
EPOCHS, BATCH_SIZE = 20, 128
LEARNING_RATE = 0.1
SEED = 2026
RUN_NAME = 'resnet18'
DOWNLOAD = True                   # 完整缓存存在后可改为 False
NUM_THREADS = min(4, os.cpu_count() or 1)
DATA_ROOT = LAB_ROOT / 'data'
OUTPUT_ROOT = LAB_ROOT / 'outputs'
if min(TRAIN_SIZE, VAL_SIZE, EPOCHS, BATCH_SIZE) < 1:
    raise ValueError('样本数、轮数和 batch size 必须大于 0。')
if TRAIN_SIZE + VAL_SIZE > 50000:
    raise ValueError('训练与验证样本数之和不能超过 50000。')
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME 只能包含英文字母、数字、下划线或连字符。')
if not 0 < LEARNING_RATE < float('inf'):
    raise ValueError('学习率必须为大于 0 的有限数值。')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_ROOT / f'{RUN_NAME}_best.pt'
torch.set_num_threads(NUM_THREADS)
device = select_device(DEVICE_CHOICE)
set_seed(SEED)
print('实际运行设备:', device)
print('训练 / 验证样本数:', TRAIN_SIZE, VAL_SIZE)
print('轮数 / batch size / 初始学习率:', EPOCHS, BATCH_SIZE, LEARNING_RATE)


## 3. 读取与查看 CIFAR-10

CIFAR-10 包含 50,000 张训练图和 10,000 张独立测试图，图像形状为 `3×32×32`。
本项目仅从官方训练集划分训练和验证索引；测试集不参与训练和模型选择。

训练预处理依次为随机裁剪 `RandomCrop(32, padding=4)`、随机水平翻转、
`ToTensor()` 与通道标准化。验证、测试只进行张量转换和标准化。
两个数据视图使用各自的变换，验证图像不会受到随机增强影响。
显示图像时会反标准化，再按 RGB 顺序显示。

首次读取会下载约 163 MiB 的官方压缩包并解压。
也可先在终端运行 `python prepare_data.py`，之后将 `DOWNLOAD` 改为 `False`。
即使只用少量样本，首次下载仍需要完整数据包。


In [ ]:
train_loader, val_loader = make_loaders(
    DATA_ROOT, TRAIN_SIZE, VAL_SIZE, BATCH_SIZE, SEED, DOWNLOAD, device)
train_ids = set(train_loader.dataset.indices)
val_ids = set(val_loader.dataset.indices)
assert train_ids.isdisjoint(val_ids)
images, labels = next(iter(train_loader))
print('类别:', CLASS_NAMES)
print('train / val:', len(train_loader.dataset), len(val_loader.dataset))
print('训练 / 验证索引交集:', len(train_ids & val_ids))
print('images:', tuple(images.shape), images.dtype)
print('labels:', tuple(labels.shape), labels.dtype)
print('标准化 mean / std:', CIFAR_MEAN, CIFAR_STD)
show_batch(train_loader)


## 4. 完整 ResNet-18 与单轮训练代码

残差块计算 `ReLU(F(x) + shortcut(x))`。
输入与输出的通道数或空间尺寸不同时，捷径分支使用 `1×1` 卷积与 BatchNorm 对齐。

| 部分 | 输出形状，不含 batch 维度 |
|---|---|
| `3×3` 卷积，步长 1，BatchNorm，ReLU | `64×32×32` |
| 第 1 组，2 个残差块 | `64×32×32` |
| 第 2 组，2 个残差块 | `128×16×16` |
| 第 3 组，2 个残差块 | `256×8×8` |
| 第 4 组，2 个残差块 | `512×4×4` |
| 全局平均池化与全连接层 | `10` |

首层采用 `3×3` 卷积并移除最前面的最大池化，保留小图像的空间信息。
这是适配 CIFAR-10 的 ResNet-18，与原论文的 CIFAR ResNet-20/32 属于不同配置。
网络从随机参数开始训练，不使用 ImageNet 预训练权重。
最后一层直接输出 logits，交给 `CrossEntropyLoss`。

每个 batch 完整执行清梯度、前向计算、损失计算、反向传播和参数更新。
下方源码与项目中的 `resnet_model.py` 一致。


In [ ]:
"""适配 CIFAR-10 的完整 ResNet-18 和单轮训练代码。"""
import torch
from torch import nn


class BasicBlock(nn.Module):
    """两层 3×3 卷积与残差连接；改变尺寸时使用 1×1 投影。"""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class ResNet18(nn.Module):
    """输入 [N,3,32,32]，输出 [N,10]；参数从随机初始化开始训练。"""

    def __init__(self, num_classes=10):
        super().__init__()
        self.in_channels = 64
        # CIFAR-10 保持 32×32 输入，首层使用 3×3、stride=1，并省去初始最大池化。
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def _make_layer(self, channels, blocks, stride):
        layers = [BasicBlock(self.in_channels, channels, stride)]
        self.in_channels = channels
        layers.extend(BasicBlock(channels, channels) for _ in range(1, blocks))
        return nn.Sequential(*layers)

    def forward(self, images):
        features = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(images)))))
        # CrossEntropyLoss 直接接收 logits，网络末尾无需 Softmax。
        return self.fc(torch.flatten(self.avgpool(features), 1))


def train_one_epoch(model, loader, criterion, optimizer, device):
    """执行一轮训练，按实际样本数计算平均损失与准确率。"""
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for step, (images, labels) in enumerate(loader, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
        if step % 50 == 0 or step == len(loader):
            print(f"  batch {step}/{len(loader)} | loss {total_loss / total:.4f} "
                  f"| accuracy {total_correct / total:.2%}", flush=True)
    if total == 0:
        raise ValueError("训练数据为空。")
    return {"loss": total_loss / total, "acc": total_correct / total}


In [ ]:
set_seed(SEED)
preview_model = ResNet18().to(device)
preview_model.eval()
with torch.no_grad():
    logits = preview_model(images[:4].to(device))
print(preview_model)
print('可训练参数量:', count_parameters(preview_model))
print('输入:', tuple(images[:4].shape), '| 输出:', tuple(logits.shape))
assert tuple(logits.shape) == (min(4, images.size(0)), 10)
del preview_model, logits


## 5. 训练、验证与保存

优化器为 SGD，动量 `0.9`、权重衰减 `5e-4`，初始学习率 `0.1`。
使用余弦退火学习率调度，每轮结束更新学习率。
每轮训练结束后，只在验证集评估；验证准确率提高时保存参数，相同则保留较早轮次。

`model.train()` 会让 BatchNorm 使用当前 batch 的统计量并更新运行统计。
`model.eval()` 让 BatchNorm 使用已保存的运行统计，验证与测试同时关闭梯度记录。
模型权重和 BatchNorm 的缓冲量一并保存，训练结束时恢复验证表现最佳的一组参数。

每次运行本训练单元都会重新初始化网络和优化器。
同名输出会被覆盖，需要保留结果时修改 `RUN_NAME`；本流程不提供中断后续训。


In [ ]:
# 重建加载器和模型，使图像预览不改变正式训练的随机状态与批顺序。
set_seed(SEED)
train_loader, val_loader = make_loaders(
    DATA_ROOT, TRAIN_SIZE, VAL_SIZE, BATCH_SIZE, SEED, DOWNLOAD, device)
model = ResNet18().to(device)
config = {
    'run_name': RUN_NAME, 'model': 'ResNet18', 'dataset': 'CIFAR-10',
    'seed': SEED, 'train_size': TRAIN_SIZE, 'val_size': VAL_SIZE,
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'lr': LEARNING_RATE,
    'optimizer': 'SGD', 'momentum': 0.9, 'weight_decay': 5e-4,
    'scheduler': 'CosineAnnealingLR', 'pretrained': False,
    'normalization': {'mean': list(CIFAR_MEAN), 'std': list(CIFAR_STD)},
    'augmentation': ['RandomCrop(32,padding=4)', 'RandomHorizontalFlip'],
    'device_requested': DEVICE_CHOICE, 'device': str(device),
    'data_root': str(DATA_ROOT), 'output_dir': str(OUTPUT_ROOT),
    'num_threads': NUM_THREADS, 'python_version': sys.version.split()[0],
    'torch_version': str(torch.__version__),
    'torchvision_version': str(torchvision.__version__),
    'parameters': count_parameters(model),
}
(OUTPUT_ROOT / f'{RUN_NAME}_config.json').write_text(
    json.dumps(config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
history = fit(model, train_loader, val_loader, train_one_epoch, device,
              epochs=EPOCHS, lr=LEARNING_RATE, checkpoint=checkpoint_path, seed=SEED)
(OUTPUT_ROOT / f'{RUN_NAME}_history.json').write_text(
    json.dumps(history, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
plot_history(history, save_path=OUTPUT_ROOT / f'{RUN_NAME}_curves.png')
best = max(history, key=lambda row: row['val_acc'])
print(f"最佳验证准确率: {best['val_acc']:.2%} | 轮次: {best['epoch']}")
print('模型文件:', checkpoint_path)


## 6. 加载最佳模型并测试

训练配置确定后，使用完整的 10,000 张官方测试图像进行评估。
此处重新创建网络，从硬盘加载参数，用于确认模型可以独立使用。
`map_location` 支持把训练得到的参数加载到当前设备。
测试不更新模型参数和 BatchNorm 运行统计。

混淆矩阵的行表示真实类别，列表示预测类别；对角线为预测正确的样本数。
误分类图中同时显示真实类别与预测类别。


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
final_model = ResNet18().to(device)
final_model.load_state_dict(checkpoint['model_state'])
test_loader = make_test_loader(DATA_ROOT, BATCH_SIZE, DOWNLOAD, device)
result = evaluate(final_model, test_loader, nn.CrossEntropyLoss(), device,
                  return_predictions=True)
test_metrics = {
    'checkpoint': str(checkpoint_path), 'device': str(device),
    'test_samples': len(test_loader.dataset),
    'test_loss': result['loss'], 'test_accuracy': result['acc'],
    'best_epoch': checkpoint['epoch'], 'best_val_accuracy': checkpoint['val_acc'],
    'train_size': checkpoint['train_size'], 'val_size': checkpoint['val_size'],
    'parameters': count_parameters(final_model), 'class_names': CLASS_NAMES,
    'torch_version': str(torch.__version__),
}
(OUTPUT_ROOT / f'{RUN_NAME}_test_metrics.json').write_text(
    json.dumps(test_metrics, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('最佳轮次:', checkpoint['epoch'], '| 测试样本数:', len(test_loader.dataset))
print(f"测试损失: {result['loss']:.4f} | 测试准确率: {result['acc']:.2%}")
confusion = plot_confusion(result['true'], result['pred'],
                          save_path=OUTPUT_ROOT / f'{RUN_NAME}_confusion.png')
show_mistakes(final_model, test_loader, device,
              save_path=OUTPUT_ROOT / f'{RUN_NAME}_mistakes.png')


## 7. 查看与保留结果

| 默认文件 | 内容 |
|---|---|
| `outputs/resnet18_best.pt` | 最佳模型参数与 BatchNorm 运行统计 |
| `outputs/resnet18_config.json` | 运行配置 |
| `outputs/resnet18_history.json` | 每轮损失、准确率、学习率与耗时 |
| `outputs/resnet18_curves.png` | 训练与验证曲线 |
| `outputs/resnet18_test_metrics.json` | 独立测试结果 |
| `outputs/resnet18_confusion.png` | 测试集混淆矩阵 |
| `outputs/resnet18_mistakes.png` | 误分类样例 |

使用其他 `RUN_NAME` 时，文件前缀相应改变。
检查训练损失的总体趋势、验证准确率，以及模型重新加载后的测试结果。
增加训练轮数、样本数或调整学习率时，根据验证集表现选择配置。
不同硬件、软件版本及随机过程可能带来数值差异，结果以实际运行为准。

| 项目 | 实际结果 |
|---|---|
| 运行设备 | |
| 训练、验证、测试样本数 | |
| 轮数、batch size、初始学习率 | |
| 最佳验证准确率与轮次 | |
| 测试准确率 | |
| 模型与曲线路径 | |

保存 Notebook 输出、模型参数、配置和曲线，即可复查本次运行。
